# Hierarchical clustering and Gaussian mixtures

Both of these methods find **groups in data without being told the answers**. This is called **clustering**, and it is a kind of **unsupervised learning**.

- **Hierarchical clustering** joins the closest points together, one step at a time, and builds a **tree** of groups.
- **Gaussian mixtures** look for **hills** in a pile of measurements, and say how likely each point is to belong to each hill.

**How to use this notebook**

- Run each cell in order with **Shift + Enter**.
- Cells marked **Exercise** need you to do something. Most just need you to replace `___` with the right value, or answer a question in a text cell.
- Some cells check your answer for you and print *Correct* or *Not quite*.
- If something breaks, use **Kernel > Restart & Run All** and work down from the top again.

## Key words

| Word | Meaning |
|---|---|
| **Cluster** | A group of points that are close to each other |
| **Distance** | How far apart two points are, measured in a straight line |
| **Dendrogram** | A tree diagram that shows the order in which groups were joined |
| **Cutting the tree** | Drawing a line across the dendrogram to choose how many groups we want |
| **Hill** | A pile of measurements bunched around a middle value. Also called a **bell curve** or **Gaussian** |
| **Gaussian mixture** | Two or more hills added together |

## Setting up

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.mixture import GaussianMixture

import warnings
warnings.filterwarnings("ignore")   # hide technical warnings so the output is easier to read

# Part 1: Hierarchical clustering

We start with just **6 points**, called A to F, so you can follow every step.

In [ ]:
names = ["A", "B", "C", "D", "E", "F"]
points = np.array([[1.0, 1.0],
                   [1.7, 1.2],
                   [1.3, 2.4],
                   [5.0, 4.0],
                   [5.6, 4.5],
                   [4.6, 5.6]])

plt.figure(figsize=(5, 5))
plt.scatter(points[:, 0], points[:, 1], s=150, color="black")
for name, (x, y) in zip(names, points):
    plt.text(x + 0.15, y + 0.15, name, fontsize=14, fontweight="bold")
plt.xlim(0, 7)
plt.ylim(0, 7)
plt.grid(alpha=0.3)
plt.show()

### Exercise 1: think first

Look at the chart. Without writing any code:

a) How many groups can you see?

b) Which **two points** look closest together? These will be joined first.

*Write your answer here.*

### Exercise 2: measure a distance by hand

The straight-line distance between two points uses Pythagoras:

> distance = √( (difference in x)² + (difference in y)² )

A is at (1.0, 1.0) and B is at (1.7, 1.2). Fill in the blank to work out the distance between them. `np.sqrt` takes a square root and `**2` squares a number.

In [ ]:
distance_AB = np.sqrt((___ - 1.0)**2 + (___ - 1.0)**2)
print("Distance from A to B:", round(distance_AB, 2))

In [ ]:
# Check cell: just run this
if abs(distance_AB - 0.728) < 0.01:
    print("Correct!")
else:
    print("Not quite. A is at (1.0, 1.0) and B is at (1.7, 1.2).")

Doing that by hand for every pair would take a while, so here is a table of **every** distance. Read it like a mileage chart: the row is one point, the column is the other.

In [ ]:
distances = np.zeros((6, 6))
for i in range(6):
    for j in range(6):
        distances[i, j] = np.sqrt(((points[i] - points[j]) ** 2).sum())

pd.DataFrame(distances, index=names, columns=names).round(2)

### Exercise 3: read the table

a) Apart from the 0s, what is the **smallest** distance in the table, and which two points does it belong to?

b) What is the **second smallest**?

c) Why are there 0s running down the middle of the table?

*Write your answer here.*

### Exercise 4: let the computer do the joining

SciPy's `linkage` function does all the joins for us. Fill in the blank with `"single"`. Single linkage means the distance between two groups is the distance between their **closest** pair of points.

The rest of the cell just prints each join in plain English.

In [ ]:
joins = linkage(points, method=___)

groups = {i: names[i] for i in range(6)}      # each point starts as its own group
for step, (a, b, dist, size) in enumerate(joins, start=1):
    new_group = "".join(sorted(groups[int(a)] + groups[int(b)]))
    print(f"Step {step}: join {groups[int(a)]:>3} + {groups[int(b)]:<3}  at distance {dist:.2f}")
    groups[6 + step - 1] = new_group

Six points took **five** joins to end up as one big group. How many joins would **10** points need? Can you spot the rule?

*Write your answer here.*

## The tree (dendrogram)

A **dendrogram** shows every join as a tree. Each point is a leaf at the bottom, each flat bar is a join, and the **height** of the bar is the distance at which the join happened.

In [ ]:
plt.figure(figsize=(8, 5))
dendrogram(joins, labels=names, color_threshold=0, above_threshold_color="black")
plt.ylabel("Distance when joined")
plt.show()

### Exercise 5: read the tree

a) At what height do the two big groups (ABC and DEF) finally join?

b) Look at the gap between the last two joins. Why does a **big gap** suggest the data really has two groups?

*Write your answer here.*

### Exercise 6: cut the tree

To get groups, we **cut** the tree with a horizontal line at a chosen height. Every vertical line it crosses becomes one group.

Fill in the blank with a cut height of **2.5**, which is in the big gap.

In [ ]:
cut_height = ___

plt.figure(figsize=(8, 5))
dendrogram(joins, labels=names, color_threshold=0, above_threshold_color="black")
plt.axhline(cut_height, color="red", linestyle="--", linewidth=2)
plt.ylabel("Distance when joined")
plt.show()

labels = fcluster(joins, t=cut_height, criterion="distance")
print("Number of groups:", len(set(labels)))
for group in sorted(set(labels)):
    members = [n for n, l in zip(names, labels) if l == group]
    print(f"Group {group}: {', '.join(members)}")

In [ ]:
# Check cell: just run this
if len(set(labels)) == 2:
    print("Correct! Two groups.")
else:
    print("Not quite. Set cut_height to 2.5 and run the cell above again.")

### Exercise 7: cut lower

Copy the cell from Exercise 6 into the empty cell below, and change the cut height to **1.0**.

How many groups do you get now, and which points are in each one?

In [ ]:
# Paste your code here and change the cut height to 1.0

*Write your answer here.*

# Part 2: Gaussian mixtures

Now for a different kind of data. We have a box of **200 apples** and we weigh every one. This is **simulated data**: we made up 120 apples of one kind and 80 of another, and mixed them in the same box.

The model will **not** be told which kind each apple is. It only sees the weights.

In [ ]:
rng = np.random.default_rng(4)      # fixed seed, so everyone gets the same apples

kind_1 = rng.normal(150, 12, 120)   # 120 apples, around 150 g
kind_2 = rng.normal(190, 16, 80)    # 80 apples, around 190 g
all_apples = np.concatenate([kind_1, kind_2])

print("Apples in the box:", len(all_apples))
print("First ten weights:", all_apples[:10].round(1))

In [ ]:
bins = np.arange(100, 255, 5)       # weight ranges: 100-105 g, 105-110 g, ...

plt.figure(figsize=(9, 4))
plt.hist(all_apples, bins=bins, color="grey", edgecolor="white")
plt.xlabel("Apple weight (grams)")
plt.ylabel("Number of apples")
plt.title("All 200 apples")
plt.show()

### Exercise 8: think first

Look at the chart. Each bar shows how many apples weigh about that much.

a) How many **hills** (bumps) can you see?

b) Roughly what weight is the **middle** of each hill?

c) Where would you be **least sure** which kind an apple is?

*Write your answer here.*

### Exercise 9: fit a Gaussian mixture

`GaussianMixture` looks for hills in the data. We have to tell it **how many** hills to look for. Fill in the blank.

(scikit-learn wants the weights as a single column, which is what `reshape(-1, 1)` does.)

In [ ]:
X = all_apples.reshape(-1, 1)

model = GaussianMixture(n_components=___, random_state=0)
model.fit(X)

# sort the hills from lightest to heaviest, so hill 1 is always the lighter one
order = np.argsort(model.means_.ravel())
middles = model.means_.ravel()[order]
widths = np.sqrt(model.covariances_.ravel()[order])
sizes = model.weights_[order]

for i in range(len(middles)):
    print(f"Hill {i + 1}: middle {middles[i]:.0f} g, width {widths[i]:.0f} g, holds {sizes[i]:.0%} of the apples")

In [ ]:
# Check cell: just run this
if len(middles) == 2 and abs(middles[0] - 150) < 5 and abs(middles[1] - 190) < 8:
    print("Correct! The model found two hills.")
else:
    print("Not quite. How many hills did you see in Exercise 8?")

Each hill is described by three numbers:

- **middle**: the typical weight
- **width**: how spread out the weights are (a wider hill means more variety)
- **size**: what share of all the apples belong to that hill

Here are the two hills drawn over the apples.

In [ ]:
def bell(x, middle, width):
    return np.exp(-0.5 * ((x - middle) / width) ** 2) / (width * np.sqrt(2 * np.pi))

weights = np.linspace(100, 250, 300)

plt.figure(figsize=(9, 4))
plt.hist(all_apples, bins=bins, color="lightgrey", edgecolor="white")
plt.plot(weights, bell(weights, middles[0], widths[0]) * sizes[0] * 200 * 5, color="tab:blue", linewidth=3, label="Hill 1")
plt.plot(weights, bell(weights, middles[1], widths[1]) * sizes[1] * 200 * 5, color="tab:orange", linewidth=3, label="Hill 2")
plt.xlabel("Apple weight (grams)")
plt.ylabel("Number of apples")
plt.legend()
plt.show()

Compare the numbers the model found with the code that made the apples (`rng.normal(150, 12, 120)` and `rng.normal(190, 16, 80)`). How close did it get? Remember, it was never told which apple was which.

*Write your answer here.*

### Exercise 10: which hill does an apple belong to?

The big strength of a Gaussian mixture is that it gives **percentages**, not just a yes or no. For each apple it compares how tall each hill is at that weight.

Three test apples are already in the list. Add a fourth apple weighing **165 g** by filling in the blank.

In [ ]:
test_apples = np.array([[140.0], [170.0], [200.0], [___]])

percentages = model.predict_proba(test_apples)[:, order]
for apple, (p1, p2) in zip(test_apples.ravel(), percentages):
    print(f"{apple:.0f} g apple:  hill 1 {p1:6.1%}   hill 2 {p2:6.1%}")

Which of the four apples is the model **least sure** about? Why does that make sense when you look at the chart of the two hills?

*Write your answer here.*

### Exercise 11: how many did it get right?

Because we made up the apples, we know which kind each one really is. (With real data we usually would not.) The first 120 apples are kind 1 and the last 80 are kind 2.

Fill in the blank to count how many apples the model put in the right group.

In [ ]:
true_kind = np.array([1] * 120 + [2] * 80)

hill = model.predict(X)                       # most likely hill for each apple
hill = np.where(hill == order[0], 1, 2)       # name the lighter hill 1 and the heavier hill 2

correct = (hill == ___).sum()
print("Apples put in the right group:", correct, "out of", len(true_kind))

In [ ]:
mistakes = all_apples[hill != true_kind]
print("Weights of the apples it got wrong:", np.sort(mistakes).round(0))

Look at the weights of the apples it got wrong. Why do you think the model made these mistakes?

*Write your answer here.*

### Exercise 12 (bonus): the wrong number of hills

What happens if we tell the model to look for **3** hills instead of 2? Fill in the blank and run the cell.

In [ ]:
model_3 = GaussianMixture(n_components=___, random_state=0).fit(X)

order_3 = np.argsort(model_3.means_.ravel())
for i, k in enumerate(order_3):
    print(f"Hill {i + 1}: middle {model_3.means_.ravel()[k]:.0f} g, holds {model_3.weights_[k]:.0%} of the apples")

The model always finds exactly the number of hills you ask for, even if that is the wrong number. One way to choose is a score called **BIC**. You do not need to know how it is worked out: just remember that **lower is better**.

In [ ]:
for k in [1, 2, 3, 4]:
    m = GaussianMixture(n_components=k, random_state=0).fit(X)
    print(f"{k} hill(s): BIC = {m.bic(X):.1f}")

Which number of hills has the lowest BIC? Does that match what you saw in Exercise 8?

*Write your answer here.*

## Summary

| | Hierarchical clustering | Gaussian mixture |
|---|---|---|
| Main idea | Keep joining the closest groups | Find hills in the data |
| What you get | A tree (dendrogram) | The middle, width and size of each hill |
| Choosing the number of groups | Cut the tree, ideally across the biggest gap | Set `n_components`, and check with BIC |
| Each point's group | One group only | A **percentage** for each group |
| Unsure points | Forced into one group | Get a split, such as 60 / 40 |

### Final question

A shop sorts its customers by how much they spend each month. A customer who sits right between two groups gets a different discount depending on which group they are put in. Which method would you choose, and why?

*Write your answer here.*